*Eksplorasi Data & Preprocessing*

Sistem rekomendasi wisata Batam berbasis Content-Based Filtering.

**Dataset:** 155 tempat wisata, kuliner, hotel, dan tempat menarik di Batam.

In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("data/destinations_minimal.csv")

In [3]:
data.head()

,wisata_id,nama,area,lat,lon,harga_bucket,rating,jumlah_ulasan,cat_alam,cat_budaya,cat_hiburan,cat_belanja,cat_kuliner,cat_wellness,cat_religi,tags,deskripsi_id,deskripsi_my,deskripsi_en
0,1001,Batam Marriot Hotel,Batu Ampar,1.15434,103.99695,3,4.8,1396,False,False,False,False,False,False,False,hotel;hiburan;kuliner;wellness,Bintang 5,NaN,NaN
1,1002,Wyndham Panbil Batam,Sei Beduk,1.08218,104.02962,2,4.6,4822,False,False,False,False,False,False,False,hotel;alam;hiburan;kuliner;wellness,Bintang 5,NaN,NaN
2,1003,Montigo Resort,Nongsa,1.18068,104.07436,2,4.4,3141,False,False,False,False,False,False,False,hotel;alam;hiburan;kuliner;wellness,Bintang 5,NaN,NaN
3,1004,Radisson Golf & Convention Centre Batam,Batam Kota,1.10353,104.03180,2,4.6,4441,False,False,False,False,False,False,False,hotel;hiburan;kuliner;wellness,Bintang 5,NaN,NaN
4,1005,Oakwood Hotel & Apartments Grand Batam,Lubuk Baja,1.13557,104.00735,2,4.6,190,False,False,False,False,False,False,False,hotel;alam;hiburan;belanja;kuliner,Bintang 4,NaN,NaN


In [4]:
data.shape

(154, 19)

In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 154 entries, 0 to 153
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   wisata_id      154 non-null    int64  
 1   nama           154 non-null    str    
 2   area           154 non-null    str    
 3   lat            154 non-null    float64
 4   lon            154 non-null    float64
 5   harga_bucket   154 non-null    int64  
 6   rating         154 non-null    float64
 7   jumlah_ulasan  154 non-null    int64  
 8   cat_alam       154 non-null    bool   
 9   cat_budaya     154 non-null    bool   
 10  cat_hiburan    154 non-null    bool   
 11  cat_belanja    154 non-null    bool   
 12  cat_kuliner    154 non-null    bool   
 13  cat_wellness   154 non-null    bool   
 14  cat_religi     154 non-null    bool   
 15  tags           154 non-null    str    
 16  deskripsi_id   68 non-null     str    
 17  deskripsi_my   0 non-null      float64
 18  deskripsi_en   0 non-

## Masalah pada Data Mentah


### 1. Kolom kategori (cat_*) semua kosong

In [6]:
cat_cols = [c for c in data.columns if c.startswith("cat_")]
data[cat_cols].sum()

cat_alam        0
cat_budaya      0
cat_hiburan     0
cat_belanja     0
cat_kuliner     0
cat_wellness    0
cat_religi      0
dtype: int64

Semua nol. Kategori asli ternyata ada di kolom `tags`.

In [7]:
data["tags"].head(10)

0         hotel;hiburan;kuliner;wellness
1    hotel;alam;hiburan;kuliner;wellness
2    hotel;alam;hiburan;kuliner;wellness
3         hotel;hiburan;kuliner;wellness
4     hotel;alam;hiburan;belanja;kuliner
5         hotel;hiburan;kuliner;wellness
6         hotel;hiburan;kuliner;wellness
7    hotel;alam;hiburan;kuliner;wellness
8                          hotel;kuliner
9                  hotel;hiburan;kuliner
Name: tags, dtype: str

### 2. Deskripsi banyak yang kosong / tidak berguna

In [8]:
data["deskripsi_id"].value_counts().head(10)

deskripsi_id
Bintang 4                                                                           25
Bintang 3                                                                           14
Pantai Tradisional                                                                  14
Bintang 5                                                                            4
Mall Modern, brand National dan International, elektronik, foodcourt                 3
Mall Modern, brand internasional, Lokasi dekat Ferry Terminal Batam Center           1
Mall modern, brand internasional, lokasi dekat Pusat Kota                            1
Mall Modern, Brand Nasional, handphones murah, elektronik, fashion, shopping dll     1
Mall modern, brand Nasional, permainan, food court, franchise resto                  1
Mall Modern, Brand Nasional dan International, fashion, shopping, Makanandll         1
Name: count, dtype: int64

Banyak yang isinya cuma "Bintang 5", bukan deskripsi sebenarnya.

## Distribution Label
Kategori diambil dari tag pertama tiap tempat.

In [9]:
data["tag1"] = data["tags"].str.split(";").str[0]
data["tag1"].value_counts()

tag1
wisata_kuliner        53
hotel                 43
pariwisata            25
wisata_religi         12
pusat_perbelanjaan    10
alam                  10
religi                 1
Name: count, dtype: int64

In [10]:
cat_map = {"hotel":"Hotel","wisata_kuliner":"Kuliner","pariwisata":"Wisata",
           "wisata_religi":"Religi","religi":"Religi","pusat_perbelanjaan":"Belanja","alam":"Alam"}
data["kategori"] = data["tag1"].map(cat_map)
# Museum = atraksi wisata/edukasi (meski bertema budaya/religi)
data.loc[data["nama"].str.contains("Museum", case=False, na=False), "kategori"] = "Wisata"
data["kategori"].value_counts()

kategori
Kuliner    53
Hotel      43
Wisata     26
Religi     12
Belanja    10
Alam       10
Name: count, dtype: int64

### Visualisasi distribusi

In [11]:
import matplotlib.pyplot as plt

data["kategori"].value_counts().plot(kind="bar", color="steelblue")
plt.title("Jumlah Tempat per Kategori")
plt.ylabel("Jumlah")
plt.show()

C:\Users\Hen\AppData\Local\Temp\ipykernel_27176\1184700108.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
data["area"].value_counts().plot(kind="barh", color="teal")
plt.title("Jumlah Tempat per Area")
plt.show()

C:\Users\Hen\AppData\Local\Temp\ipykernel_27176\2332851126.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Data Cleaning

### 1. Perbaiki encoding nama (Café yang rusak)

In [13]:
data["nama"] = data["nama"].str.replace("â€š", "é", regex=False).str.strip()

### 2. Perbaiki koordinat error
Ada 1 tempat dengan latitude 110420 (harusnya 1.1042 - salah titik desimal).

In [14]:
data[data["lat"] > 100][["nama","lat","lon"]]

,nama,lat,lon
77,Batam Adventure Park,110420.0,104.03727


In [15]:
data.loc[data["lat"] > 100, "lat"] = data.loc[data["lat"] > 100, "lat"] / 100000

### 3. Tambah durasi kunjungan per kategori
Data asli tidak punya durasi, jadi diisi default yang masuk akal (menit).

In [16]:
durasi = {"Kuliner":75,"Belanja":120,"Wisata":90,"Alam":90,"Religi":45,"Hotel":0}
data["durasi_menit"] = data["kategori"].map(durasi)

### 4. Buat deskripsi
Tempat terkenal diberi deskripsi manual yang akurat. Sisanya dibuat otomatis dari kata kunci pada nama + area + tags, supaya teksnya kaya untuk pencarian.

In [17]:
import re

# deskripsi manual untuk tempat-tempat terkenal
overrides = {
 1001:"Hotel bintang 5 mewah di Batu Ampar dengan fasilitas lengkap dekat pelabuhan.",
 1006:"Hotel bintang 4 di kawasan Nagoya, pusat keramaian Batam, dekat pusat belanja dan kuliner.",
 1044:"Mall modern terbesar di Batam Center dengan brand internasional, bioskop, dan food court.",
 1050:"Mall ikonik dan terbesar di Nagoya dengan brand internasional, bioskop, dan pusat hiburan.",
 1073:"Jembatan ikonik kebanggaan Batam di kawasan Barelang, spot wisata dan foto favorit dengan pemandangan laut.",
 1101:"Restoran seafood ikonik di Bengkong yang menyajikan hidangan laut segar di tepi pantai.",
 1104:"Kedai mie pedas populer harga ekonomis di Bengkong, terkenal dengan mie aneka level pedas dan dimsum.",
 1122:"Salah satu vihara Buddha terbesar di Asia Tenggara di Batam Kota, dengan patung megah dan area ibadah luas.",
 1124:"Masjid ikonik berbentuk tanjak Melayu di Nongsa dengan arsitektur unik dan pemandangan laut.",
 1068:"Situs sejarah bekas kamp pengungsi Vietnam di Pulau Galang, kini jadi wisata sejarah.",
 1139:"Foodcourt / pujasera populer di kawasan Penuin (Batu Selicin), Lubuk Baja, dekat BCS Mall. Banyak kios makanan, seafood, dan hidangan kaki lima, buka dari pagi hingga malam.",
 1140:"Restoran seafood tepi laut di kawasan Harbour Bay, Batu Ampar, Batam. ikan, kepiting, dan udang segar, dekat pelabuhan feri ke Singapura.",
 1141:"Restoran seafood / kelong tepi laut di Nongsa, Batam. hidangan ikan dan laut dengan pemandangan pantai.",
 1142:"Kafe tempat ngopi santai di kawasan Batu Aji (Sagulung), Batam. kopi dan tempat nongkrong.",
 1143:"Restoran seafood Melayu / kelong tepi laut di Sekupang, Batam. ikan segar dengan pemandangan laut dan sunset.",
 1144:"Kafe kopi dan toast (Janji Jiwa) di Tiban, Sekupang, Batam. tempat ngopi santai.",
 1145:"Kafe tempat ngopi dengan live music di Bengkong, Batam. tempat nongkrong santai indoor dan outdoor.",
 1146:"Kafe kopi populer di Nagoya (Lubuk Baja), Batam. tempat ngopi santai dan nyaman.",
 1147:"Kafe dan floating bar tepi laut di Bengkong, Batam. tempat nongkrong santai dengan pemandangan laut dan live music.",
 1148:"Kafe tempat ngopi dan brunch di Batam Kota, Batam. tempat nongkrong santai dengan suasana nyaman.",
 1149:"Kafe kopi di Tiban, Sekupang, Batam. tempat ngopi santai untuk nongkrong.",
 1150:"Kafe dan bistro tempat nongkrong di Batam Kota, Batam. live music dan tempat santai malam hari.",
 1151:"Kafe dan resto tempat nongkrong di Batam Kota, Batam. banyak pilihan makanan, dessert, dan tempat foto.",
 1152:"Pujasera / foodcourt di kawasan Batamindo, Muka Kuning (Sei Beduk), Batam. banyak pilihan makanan dengan harga murah.",
 1153:"Wahana wisata air / waterpark di kawasan Mall Top 100, Batu Aji, Batam. kolam dan seluncuran, cocok untuk keluarga dan anak.",
 1154:"Spot wisata bukit dengan pemandangan kota dan laut di Sekupang, Batam. populer untuk foto dan menikmati sunset.",
}

# kata kunci dari nama untuk perkaya deskripsi
def keywords(nama):
    n = nama.lower(); k = []
    if any(w in n for w in ["seafood","prawn","ikan bakar","kelong"]): k.append("seafood hidangan laut segar")
    if any(w in n for w in ["coffee","coffe","kopi","cafe","café"]): k.append("kafe tempat ngopi santai")
    if any(w in n for w in ["bakery","patisserie"]): k.append("roti kue bakery")
    if "mie" in n: k.append("mie")
    if "beach" in n: k.append("beach club tepi pantai santai")
    if "pantai" in n: k.append("pantai pasir laut pemandangan")
    if any(w in n for w in ["golf","driving range"]): k.append("lapangan golf olahraga area hijau")
    if "masjid" in n: k.append("masjid tempat ibadah wisata religi")
    if any(w in n for w in ["vihara","klenteng"]): k.append("vihara klenteng tempat ibadah")
    if "mall" in n: k.append("mall pusat perbelanjaan belanja fashion")
    if any(w in n for w in ["hotel","resort","inn"]): k.append("hotel penginapan menginap")
    return ", ".join(k)

def buat_deskripsi(r):
    if r["wisata_id"] in overrides:
        return overrides[r["wisata_id"]]
    base = f"Tempat {r['kategori'].lower()} di {r['area']}, Batam"
    kw = keywords(r["nama"])
    return base + (". " + kw if kw else "") + "."

data["deskripsi"] = data.apply(buat_deskripsi, axis=1)
data[["nama","deskripsi"]].head()

,nama,deskripsi
0,Batam Marriot Hotel,Hotel bintang 5 mewah di Batu Ampar dengan fas...
1,Wyndham Panbil Batam,"Tempat hotel di Sei Beduk, Batam."
2,Montigo Resort,"Tempat hotel di Nongsa, Batam. hotel penginapa..."
3,Radisson Golf & Convention Centre Batam,"Tempat hotel di Batam Kota, Batam. lapangan go..."
4,Oakwood Hotel & Apartments Grand Batam,"Tempat hotel di Lubuk Baja, Batam. hotel pengi..."


### 5. Buat kolom search_text
Gabungan nama + area + kategori + tags + deskripsi. Ini yang nanti dicocokkan dengan input user di sistem rekomendasi.

In [18]:
data["tags_natural"] = data["tags"].str.replace("_"," ").str.replace(";"," ")
data["search_text"] = (data["nama"] + " " + data["area"] + " " + data["kategori"] + " "
                       + data["tags_natural"] + " " + data["deskripsi"])
data["search_text"].iloc[0]

'Batam Marriot Hotel Batu Ampar Hotel hotel hiburan kuliner wellness Hotel bintang 5 mewah di Batu Ampar dengan fasilitas lengkap dekat pelabuhan.'

### 6. Simpan dataset bersih

In [19]:
clean = data[["wisata_id","nama","area","kategori","lat","lon","harga_bucket",
               "rating","jumlah_ulasan","durasi_menit","deskripsi","search_text"]]
clean.columns = ["Place_Id","Place_Name","Area","Category","Lat","Long","Harga_Bucket",
                 "Rating","Jumlah_Ulasan","Durasi_Menit","Description","Search_Text"]
clean.to_csv("data/cleaned/batam_cleaned.csv", index=False)
clean.shape

(154, 12)